<a href="https://colab.research.google.com/github/gouthamsankar-clg/BIS_Lab/blob/main/Lab_5/GreyWolfOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd

def portfolio_objective(w, mu, Sigma, lam):
    portfolio_variance = w.T @ Sigma @ w
    expected_return = mu @ w
    return portfolio_variance - (lam * expected_return)

def enforce_constraints(w, lo, hi):
    # FIXED: Strict iterative allocation to ensure upper bounds are never breached
    n = len(w)
    w_fixed = np.clip(w, lo, hi)

    for _ in range(n):
        current_sum = np.sum(w_fixed)
        if np.isclose(current_sum, 1.0):
            break

        diff = 1.0 - current_sum
        adjustable = (w_fixed > lo) & (w_fixed < hi)

        if not np.any(adjustable):
            w_fixed = np.clip(w_fixed + diff / n, lo, hi)
            break

        w_fixed[adjustable] += diff / np.sum(adjustable)
        w_fixed = np.clip(w_fixed, lo, hi)

    return w_fixed

def run_grey_wolf(returns_matrix, lam, lo, hi, budget, n_wolves=30, max_iter=200):
    mu = np.mean(returns_matrix, axis=0)
    Sigma = np.cov(returns_matrix, rowvar=False)
    n_assets = returns_matrix.shape[1]

    pack = np.array([enforce_constraints(np.random.rand(n_assets), lo, hi) for _ in range(n_wolves)])

    alpha_pos, beta_pos, delta_pos = np.zeros(n_assets), np.zeros(n_assets), np.zeros(n_assets)
    alpha_score, beta_score, delta_score = np.inf, np.inf, np.inf

    for t in range(max_iter):
        for i in range(n_wolves):
            pack[i] = enforce_constraints(pack[i], lo, hi)
            fitness = portfolio_objective(pack[i], mu, Sigma, lam)

            if fitness < alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = fitness, pack[i].copy()
            elif fitness < beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = fitness, pack[i].copy()
            elif fitness < delta_score:
                delta_score, delta_pos = fitness, pack[i].copy()

        a = 2 - 2 * (t / max_iter)

        for i in range(n_wolves):
            new_wolf_pos = np.zeros(n_assets)
            for d in range(n_assets):
                r1, r2 = np.random.rand(), np.random.rand()
                A1, C1 = 2 * a * r1 - a, 2 * r2
                X1 = alpha_pos[d] - A1 * np.abs(C1 * alpha_pos[d] - pack[i, d])

                r1, r2 = np.random.rand(), np.random.rand()
                A2, C2 = 2 * a * r1 - a, 2 * r2
                X2 = beta_pos[d] - A2 * np.abs(C2 * beta_pos[d] - pack[i, d])

                r1, r2 = np.random.rand(), np.random.rand()
                A3, C3 = 2 * a * r1 - a, 2 * r2
                X3 = delta_pos[d] - A3 * np.abs(C3 * delta_pos[d] - pack[i, d])

                new_wolf_pos[d] = (X1 + X2 + X3) / 3
            pack[i] = new_wolf_pos

    w_star = enforce_constraints(alpha_pos, lo, hi)
    return w_star, budget * w_star, mu @ w_star, np.sqrt(w_star.T @ Sigma @ w_star)

# --- REPRODUCIBLE DATA GENERATION ---
np.random.seed(42)
days = 252
sample_returns = np.hstack([
    np.random.normal(0.0015, 0.020, (days, 1)),
    np.random.normal(0.0002, 0.005, (days, 1)),
    np.random.normal(0.0008, 0.012, (days, 1)),
    np.random.normal(0.0004, 0.014, (days, 1))
])

print(">>> GREY WOLF OPTIMIZER INPUT DECK SETUP <<<")
TOTAL_BUDGET = float(input("Enter Total Budget Cash Pool: "))
LAMBDA_PENALTY = float(input("Enter Risk Penalty Lambda Dial: "))
MIN_BOUND = float(input("Enter Minimum Asset Bound Floor: "))
MAX_BOUND = float(input("Enter Maximum Asset Bound Cap: "))

gwo_w, gwo_cash, gwo_ret, gwo_risk = run_grey_wolf(sample_returns, LAMBDA_PENALTY, MIN_BOUND, MAX_BOUND, TOTAL_BUDGET)

labels = ['Asset 0', 'Asset 1', 'Asset 2', 'Asset 3']
output_df = pd.DataFrame({'GWO Weight (%)': np.round(gwo_w * 100, 2), 'GWO Cash ($)': np.round(gwo_cash, 2)}, index=labels)
print("\n" + "="*50 + "\n             GREY WOLF OPTIMIZER RESULTS\n" + "="*50)
print(output_df.to_string())
print(f"-" * 50 + f"\nAnnual Return: {gwo_ret*days*100:.2f}%\nAnnual Risk Volatility: {gwo_risk*np.sqrt(days)*100:.2f}%\n" + "="*50)


>>> GREY WOLF OPTIMIZER INPUT DECK SETUP <<<
Enter Total Budget Cash Pool: 100000
Enter Risk Penalty Lambda Dial: 0.12
Enter Minimum Asset Bound Floor: 0.0
Enter Maximum Asset Bound Cap: 0.85

             GREY WOLF OPTIMIZER RESULTS
         GWO Weight (%)  GWO Cash ($)
Asset 0           17.73      17732.18
Asset 1            5.46       5463.00
Asset 2            0.00          0.00
Asset 3           76.80      76804.81
--------------------------------------------------
Annual Return: 55.01%
Annual Risk Volatility: 16.94%
